#### Грузим данные

In [4]:
import pandas as pd

listing_path = "initial_data/listings.csv.gz"
listing_df = pd.read_csv(listing_path)

In [5]:
listing_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 81853 entries, 0 to 81852
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            81853 non-null  int64  
 1   listing_url                                   81853 non-null  str    
 2   scrape_id                                     81853 non-null  int64  
 3   last_scraped                                  81853 non-null  str    
 4   source                                        81853 non-null  str    
 5   name                                          81853 non-null  str    
 6   description                                   79140 non-null  str    
 7   neighborhood_overview                         39600 non-null  str    
 8   picture_url                                   81851 non-null  str    
 9   host_id                                       81853 non-null  int64  
 1

#### Удаляем ненужные столбцы

In [6]:
listing_df2 = listing_df.copy()

# Определяем список колонок к удалению
drop_list = []
for col in listing_df.columns:

    # Добавляем в список полностью пустые колонки
    if listing_df[col].isnull().mean() == 1:
        drop_list += [col]
        continue

    # Добавляем в список колонки с одним значением
    if listing_df[col].nunique() == 1:
        drop_list += [col]
        continue

    # Добавляем в список колонки с ссылками
    if 'url' in col:
        drop_list += [col]
        continue

# Дополнительно убираем поля
drop_list += ['host_name',  # Имя хоста. Бесполензо для модели
              'estimated_occupancy_l365d',  # Cлишком сильно может коррелировать с нашей метрикой
              'host_id',  # id хоста
              'availability_60',  # То же самое, что и таргет в 30 дн. Сильно коррелирует с таргетом
                'availability_90',  # То же самое, что и таргет в 30 дн. Сильно коррелирует с таргетом
                'availability_365',  # То же самое, что и таргет в 30 дн. Сильно коррелирует с таргетом
                'availability_eoy',  # То же самое, что и таргет в 30 дн. Сильно коррелирует с таргетом
                'id',  # id объекта
                'calendar_last_scraped',  # дата скржпинга
            ]

listing_df2.drop(columns=drop_list, inplace=True)

# Предобразуем таргет
listing_df2['availability_30'] = listing_df2['availability_30'] / 30

#### Кодируем время ответа

In [7]:
# Кодируем время ответа
responce_time_coding = {
    'within an hour': 1,
    'within a few hours': 3,
    'within a day': 24,
    'a few days or more': 72,
}
listing_df2['host_response_time'] = listing_df2['host_response_time'].replace(responce_time_coding)

# Заполняем незаполнные значения максимальной категорией
listing_df2['host_response_time'] = listing_df2['host_response_time'].fillna(72)  


#### Обратабывает отвечаемость и подтверждаемость хоста

In [8]:

for col in ['host_response_rate', 'host_acceptance_rate']:
    # Переводим из % в float
    listing_df2[col] = listing_df2[col].str.replace('%', '').apply(float) / 100

    # Заполняем пропуски 0
    listing_df2[col] = listing_df2[col].fillna(0)


#### Способы верификации хоста

In [9]:
import ast 

# Преобразуем строку в реальный список py
listing_df2['host_verifications'] = listing_df2['host_verifications'].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)

# Раскрываем список, делаем энкодинг и схлопываем до изначального размера
dummies_df = pd.get_dummies(listing_df2['host_verifications'].explode(), prefix='host_verifications').groupby(level=0).max()

# Объединяем с исходным df
listing_df2 = pd.concat([listing_df2, dummies_df], axis=1)


#### Длительность на платформе

In [10]:
# Преобразуем в даты
listing_df2['host_since'] = pd.to_datetime(listing_df2['host_since'])
listing_df2['last_scraped'] = pd.to_datetime(listing_df2['last_scraped'])

# Считаем возраст хоста на платформе
listing_df2['host_age_on_platform'] = (listing_df2['last_scraped'] - listing_df2['host_since']).dt.days / 365

listing_df2['host_age_on_platform'] = listing_df2['host_age_on_platform'].fillna(0)

# Дропаем ненужное
listing_df2.drop(columns=['host_since', 'last_scraped'], inplace=True)

#### Форматируем булевые колонки

In [11]:
bool_columns_list = [
    'host_is_superhost',
    'host_has_profile_pic',
    'host_identity_verified',
    'instant_bookable',
]

for col in bool_columns_list:

    listing_df2[col] = listing_df2[col].replace({'f': False, 't': True})

    listing_df2[col] = listing_df2[col].fillna(False)


In [19]:
listing_df2.to_pickle(r'D:\AV\Education\Master\ML\Project\prepared_data\data.pkl')
# listing_df2.to_csv(r'D:\AV\Education\Master\ML\Project\prepared_data\data.csv')